In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# 09_results.ipynb — Additional Stats Cells
# OTC017 excluded from all analyses (polymicrogyria)
# ═══════════════════════════════════════════════════════════════════════════════

# ── CELL: Imports and shared setup ────────────────────────────────────────────

import sys
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from scipy.stats import pearsonr

sys.path.insert(0, '/user_data/csimmon2/git_repos/sym_pt')
from sym_pt_params import processed_dir

BASE_DIR = Path(processed_dir)
SEL_DIR  = BASE_DIR / 'group_results' / 'selectivity'
LIU_DIR  = BASE_DIR / 'group_results' / 'liu_distinctiveness'
GEO_DIR  = BASE_DIR / 'group_results' / 'geometry'

COPE_SET   = 'differential'
CATEGORIES = ['face', 'house', 'object', 'word']
SYMMETRIC  = ['house', 'object']
ASYMMETRIC = ['face', 'word']
EXCLUDE    = ['OTC017']

PREFERRED_CTRL_HEMI = {
    'face': 'right', 'word': 'left', 'house': 'left', 'object': 'left',
}

N_BOOT = 100_000
rng_boot = np.random.default_rng(42)

# ── Shared functions ──────────────────────────────────────────────────────────

def crawford_howell(patient_val, ctrl_vals):
    ctrl_vals = np.asarray(ctrl_vals)
    ctrl_vals = ctrl_vals[np.isfinite(ctrl_vals)]
    n = len(ctrl_vals)
    if n < 3:
        return np.nan, np.nan, n
    m, s = ctrl_vals.mean(), ctrl_vals.std(ddof=1)
    if s == 0:
        return np.nan, np.nan, n
    t = (patient_val - m) / (s * np.sqrt((n + 1) / n))
    p = 2 * stats.t.sf(abs(t), df=n - 1)
    return t, p, n

def bh_fdr(pvals, alpha=0.05):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    if n == 0:
        return np.array([], dtype=bool)
    order = np.argsort(pvals)
    ranked = np.empty(n)
    ranked[order] = np.arange(1, n + 1)
    threshold = (ranked / n) * alpha
    below = pvals <= threshold
    if not below.any():
        return np.zeros(n, dtype=bool)
    cutoff = pvals[order[np.where(below)[0].max()]]
    return pvals <= cutoff

def boot_ci(vals, n_boot=N_BOOT, ci=0.95):
    vals = np.asarray(vals).ravel()
    means = np.array([np.mean(rng_boot.choice(vals, size=len(vals), replace=True))
                      for _ in range(n_boot)])
    lo = (1 - ci) / 2
    return np.percentile(means, [lo * 100, (1 - lo) * 100])

def boot_ci_diff(a, b, n_boot=N_BOOT, ci=0.95):
    diffs = np.asarray(a).ravel() - np.asarray(b).ravel()
    means = np.array([np.mean(rng_boot.choice(diffs, size=len(diffs), replace=True))
                      for _ in range(n_boot)])
    lo = (1 - ci) / 2
    return np.mean(diffs), np.percentile(means, [lo * 100, (1 - lo) * 100])

print('Setup complete. OTC017 excluded (polymicrogyria).')

Setup complete. OTC017 excluded (polymicrogyria).


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# SELECTIVITY
# ═══════════════════════════════════════════════════════════════════════════════

sel_file = SEL_DIR / 'selectivity_summary.csv'
if not sel_file.exists():
    print(f'SKIP: {sel_file} not found')
else:
    sel = pd.read_csv(sel_file)
    sel['ses_int'] = sel['ses'].astype(int)
    first = sel.groupby('sub')['ses_int'].min().reset_index().rename(columns={'ses_int': 'fs'})
    sel = sel.merge(first, on='sub')
    sel = sel[sel['ses_int'] == sel['fs']]

    ctrl_sel = sel[sel['group'] == 'control'].copy()
    otc_sel  = sel[sel['group'] == 'OTC'].copy()
    otc_sel['surgery_side'] = otc_sel['intact_hemi'].map(
        lambda h: 'right' if h == 'left' else 'left')
    # Exclude OTC017
    otc_sel = otc_sel[~otc_sel['sub'].str.contains('017')]
    # Intact hemisphere only
    otc_intact = otc_sel[
        ((otc_sel['intact_hemi'] == 'left') & (otc_sel['hemi'] == 'left')) |
        ((otc_sel['intact_hemi'] == 'right') & (otc_sel['hemi'] == 'right'))
    ]

    metric = 'sum_selec_norm'

    print('SELECTIVITY: Descriptive statistics')
    print(f'  Controls: {ctrl_sel["sub"].nunique()} subjects')
    print(f'  OTC (excl 017): {otc_intact["sub"].nunique()} subjects')
    print()

    # ── Descriptive stats ─────────────────────────────────────────────────
    print(f'{"Category":<10} {"Group":<10} {"Hemi":<8} {"M":>8} {"SD":>8} {"n":>5}')
    print('─' * 50)
    for cat in CATEGORIES:
        ref = PREFERRED_CTRL_HEMI[cat]
        cv = ctrl_sel[(ctrl_sel['category'] == cat) &
                      (ctrl_sel['hemi'] == ref)][metric].values
        print(f'{cat:<10} {"control":<10} {ref:<8} {np.mean(cv):>8.3f} {np.std(cv):>8.3f} {len(cv):>5}')
        pv = otc_intact[otc_intact['category'] == cat][metric].values
        print(f'{"":<10} {"OTC":<10} {"intact":<8} {np.mean(pv):>8.3f} {np.std(pv):>8.3f} {len(pv):>5}')
    print()

    # ── Crawford-Howell per patient × category ────────────────────────────
    print('SELECTIVITY: Crawford-Howell single-case tests')
    print(f'{"Subject":<12} {"Side":<8} {"Category":<10} {"Value":>8} {"t":>8} {"p":>8} {"sig":>5}')
    print('─' * 65)

    craw_rows = []
    for cat in CATEGORIES:
        ref = PREFERRED_CTRL_HEMI[cat]
        ctrl_vals = ctrl_sel[(ctrl_sel['category'] == cat) &
                             (ctrl_sel['hemi'] == ref)][metric].values
        for _, row in otc_intact[otc_intact['category'] == cat].iterrows():
            t, p, n = crawford_howell(row[metric], ctrl_vals)
            craw_rows.append({'subject': row['sub'], 'surgery_side': row['surgery_side'],
                              'category': cat, 'value': row[metric], 't': t, 'p': p})

    cdf = pd.DataFrame(craw_rows)
    valid = cdf['p'].notna()
    cdf['sig'] = False
    if valid.sum() > 0:
        cdf.loc[valid, 'sig'] = bh_fdr(cdf.loc[valid, 'p'].values)

    for _, r in cdf.iterrows():
        sig = '*' if r['sig'] else ''
        print(f'{r["subject"]:<12} {r["surgery_side"]:<8} {r["category"]:<10} '
              f'{r["value"]:>8.3f} {r["t"]:>8.3f} {r["p"]:>8.4f} {sig:>5}')

    n_sig = cdf['sig'].sum()
    print(f'\n  {n_sig}/{len(cdf)} significant after BH-FDR')
    print()

    # ── Symmetric vs asymmetric ───────────────────────────────────────────
    print('SELECTIVITY: Symmetric vs asymmetric (OTC, intact hemisphere)')
    sym_vals, asym_vals = [], []
    for sub in otc_intact['sub'].unique():
        sub_df = otc_intact[otc_intact['sub'] == sub]
        sv = [sub_df[sub_df['category'] == c][metric].values for c in SYMMETRIC]
        av = [sub_df[sub_df['category'] == c][metric].values for c in ASYMMETRIC]
        s = np.nanmean([v[0] for v in sv if len(v)])
        a = np.nanmean([v[0] for v in av if len(v)])
        if np.isfinite(s) and np.isfinite(a):
            sym_vals.append(s)
            asym_vals.append(a)

    sym_arr = np.array(sym_vals)
    asym_arr = np.array(asym_vals)
    m_diff, ci_diff = boot_ci_diff(sym_arr, asym_arr)
    print(f'  Symmetric M  = {sym_arr.mean():.3f}  95% CI {[f"{x:.3f}" for x in boot_ci(sym_arr)]}')
    print(f'  Asymmetric M = {asym_arr.mean():.3f}  95% CI {[f"{x:.3f}" for x in boot_ci(asym_arr)]}')
    print(f'  Diff (sym-asym) = {m_diff:.3f}  95% CI [{ci_diff[0]:.3f}, {ci_diff[1]:.3f}]')
    sig = 'significant' if ci_diff[0] > 0 or ci_diff[1] < 0 else 'not significant'
    print(f'  → {sig}')

SELECTIVITY: Descriptive statistics
  Controls: 24 subjects
  OTC (excl 017): 16 subjects

Category   Group      Hemi            M       SD     n
──────────────────────────────────────────────────
face       control    right     492.963  420.614    24
           OTC        intact    415.604  327.515    16
house      control    left      499.463  325.358    24
           OTC        intact    463.982  309.572    16
object     control    left     1942.040  847.320    24
           OTC        intact   1253.140 1006.050    16
word       control    left      164.299  199.259    24
           OTC        intact    117.733  148.081    16

SELECTIVITY: Crawford-Howell single-case tests
Subject      Side     Category      Value        t        p   sig
─────────────────────────────────────────────────────────────────
sub-004      right    face         27.766   -1.061   0.2998      
sub-008      right    face        966.202    1.079   0.2917      
sub-010      left     face         19.544   -1.080 

  Symmetric M  = 858.561  95% CI ['581.482', '1160.276']
  Asymmetric M = 266.668  95% CI ['178.577', '354.579']
  Diff (sym-asym) = 591.893  95% CI [315.602, 895.271]
  → significant


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# LIU DISTINCTIVENESS
# ═══════════════════════════════════════════════════════════════════════════════

liu_file = LIU_DIR / f'liu_distinctiveness_{COPE_SET}.csv'
if not liu_file.exists():
    print(f'SKIP: {liu_file} not found')
else:
    liu = pd.read_csv(liu_file)
    first = (liu.groupby('subject_id')['session'].min()
                .reset_index().rename(columns={'session': 'fs'}))
    liu = liu.merge(first, on='subject_id')
    liu = liu[liu['session'] == liu['fs']]

    ctrl_liu = liu[liu['status'] == 'control'].copy()
    otc_liu  = liu[(liu['group'] == 'OTC') & (liu['hemi_label'] == 'intact')].copy()
    # Exclude OTC017
    otc_liu = otc_liu[~otc_liu['subject'].str.contains('017')]

    print('LIU DISTINCTIVENESS: Descriptive statistics')
    print(f'  Controls: {ctrl_liu["subject_id"].nunique()} subjects')
    print(f'  OTC (excl 017): {otc_liu["subject"].nunique()} subjects')
    print()

    # ── Descriptive stats ─────────────────────────────────────────────────
    print(f'{"Category":<10} {"Group":<10} {"M":>8} {"SD":>8} {"n":>5}')
    print('─' * 45)
    for cat in CATEGORIES:
        ref = PREFERRED_CTRL_HEMI[cat]
        cv = ctrl_liu[(ctrl_liu['category'] == cat) &
                      (ctrl_liu['hemi_label'] == ref)]['liu_distinctiveness'].values
        print(f'{cat:<10} {"control":<10} {np.mean(cv):>8.3f} {np.std(cv):>8.3f} {len(cv):>5}')
        pv = otc_liu[otc_liu['category'] == cat]['liu_distinctiveness'].values
        print(f'{"":<10} {"OTC":<10} {np.mean(pv):>8.3f} {np.std(pv):>8.3f} {len(pv):>5}')
    print()

    # ── Crawford-Howell per patient × category ────────────────────────────
    print('LIU DISTINCTIVENESS: Crawford-Howell single-case tests')
    print(f'{"Subject":<12} {"Side":<8} {"Category":<10} {"Value":>8} {"t":>8} {"p":>8} {"sig":>5}')
    print('─' * 65)

    craw_rows = []
    for cat in CATEGORIES:
        ref = PREFERRED_CTRL_HEMI[cat]
        ctrl_vals = ctrl_liu[(ctrl_liu['category'] == cat) &
                             (ctrl_liu['hemi_label'] == ref)]['liu_distinctiveness'].values
        for _, row in otc_liu[otc_liu['category'] == cat].iterrows():
            t, p, n = crawford_howell(row['liu_distinctiveness'], ctrl_vals)
            craw_rows.append({'subject': row['subject'],
                              'surgery_side': row['surgery_side'],
                              'category': cat,
                              'value': row['liu_distinctiveness'],
                              't': t, 'p': p})

    cdf = pd.DataFrame(craw_rows)
    valid = cdf['p'].notna()
    cdf['sig'] = False
    if valid.sum() > 0:
        cdf.loc[valid, 'sig'] = bh_fdr(cdf.loc[valid, 'p'].values)

    for _, r in cdf.iterrows():
        sig = '*' if r['sig'] else ''
        print(f'{r["subject"]:<12} {r["surgery_side"]:<8} {r["category"]:<10} '
              f'{r["value"]:>8.3f} {r["t"]:>8.3f} {r["p"]:>8.4f} {sig:>5}')

    n_sig = cdf['sig'].sum()
    print(f'\n  {n_sig}/{len(cdf)} significant after BH-FDR')

    # ── Which categories have significant patients? ───────────────────────
    print('\n  Per-category summary:')
    for cat in CATEGORIES:
        cat_df = cdf[cdf['category'] == cat]
        n_s = cat_df['sig'].sum()
        n_t = len(cat_df)
        direction = 'higher' if cat_df['t'].mean() > 0 else 'lower'
        print(f'    {cat}: {n_s}/{n_t} sig, mean t={cat_df["t"].mean():.2f} '
              f'(OTC {direction} than controls)')
    print()

    # ── Symmetric vs asymmetric ───────────────────────────────────────────
    print('LIU DISTINCTIVENESS: Symmetric vs asymmetric (OTC, intact hemisphere)')
    sym_vals, asym_vals = [], []
    for sub in otc_liu['subject'].unique():
        sub_df = otc_liu[otc_liu['subject'] == sub]
        sv = [sub_df[sub_df['category'] == c]['liu_distinctiveness'].values for c in SYMMETRIC]
        av = [sub_df[sub_df['category'] == c]['liu_distinctiveness'].values for c in ASYMMETRIC]
        s = np.nanmean([v[0] for v in sv if len(v)])
        a = np.nanmean([v[0] for v in av if len(v)])
        if np.isfinite(s) and np.isfinite(a):
            sym_vals.append(s)
            asym_vals.append(a)

    if len(sym_vals) >= 2:
        sym_arr = np.array(sym_vals)
        asym_arr = np.array(asym_vals)
        m_diff, ci_diff = boot_ci_diff(sym_arr, asym_arr)
        print(f'  Symmetric M  = {sym_arr.mean():.3f}  95% CI {[f"{x:.3f}" for x in boot_ci(sym_arr)]}')
        print(f'  Asymmetric M = {asym_arr.mean():.3f}  95% CI {[f"{x:.3f}" for x in boot_ci(asym_arr)]}')
        print(f'  Diff (sym-asym) = {m_diff:.3f}  95% CI [{ci_diff[0]:.3f}, {ci_diff[1]:.3f}]')
        sig = 'significant' if ci_diff[0] > 0 or ci_diff[1] < 0 else 'not significant'
        print(f'  → {sig}')
        print(f'  (higher = less distinct; positive diff = symmetric less distinct than asymmetric)')

LIU DISTINCTIVENESS: Descriptive statistics
  Controls: 22 subjects
  OTC (excl 017): 15 subjects

Category   Group             M       SD     n
─────────────────────────────────────────────
face       control       0.710    0.380    21
           OTC           0.801    0.278    14
house      control       0.479    0.787    22
           OTC           0.552    0.632    15
object     control       1.028    0.324    22
           OTC           1.189    0.416    15
word       control       0.508    0.351    21
           OTC           0.807    0.492    12

LIU DISTINCTIVENESS: Crawford-Howell single-case tests
Subject      Side     Category      Value        t        p   sig
─────────────────────────────────────────────────────────────────
OTC004       right    face          0.424   -0.718   0.4810      
OTC008       right    face          1.108    0.999   0.3298      
OTC010       left     face          0.612   -0.246   0.8082      
OTC021       left     face          1.224    1.291   0.

/tmp/ipykernel_1573438/313663506.py:88: RuntimeWarning: Mean of empty slice
  a = np.nanmean([v[0] for v in av if len(v)])


  Symmetric M  = 0.921  95% CI ['0.759', '1.076']
  Asymmetric M = 0.817  95% CI ['0.626', '1.000']
  Diff (sym-asym) = 0.104  95% CI [-0.121, 0.352]
  → not significant
  (higher = less distinct; positive diff = symmetric less distinct than asymmetric)


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# SPATIAL ORGANIZATION — Crawford t for medial-lateral arrangement
# ═══════════════════════════════════════════════════════════════════════════════

pk_file = BASE_DIR / 'group_results' / 'peak_coords' / 'peak_coords.csv'
if not pk_file.exists():
    print(f'SKIP: {pk_file} not found')
else:
    pk = pd.read_csv(pk_file)
    pk['ses_int'] = pk['ses'].astype(int)
    first = pk.groupby('sub')['ses_int'].min().reset_index().rename(columns={'ses_int': 'fs'})
    pk = pk.merge(first, on='sub')
    pk = pk[pk['ses_int'] == pk['fs']]

    ctrl_pk = pk[pk['group'] == 'control'].copy()
    otc_pk  = pk[pk['group'] == 'OTC'].copy()
    otc_pk  = otc_pk[~otc_pk['sub'].str.contains('017')]
    otc_pk  = otc_pk[
        ((otc_pk['intact_hemi'] == 'left') & (otc_pk['hemi'] == 'left')) |
        ((otc_pk['intact_hemi'] == 'right') & (otc_pk['hemi'] == 'right'))
    ]
    otc_pk['surgery_side'] = otc_pk['intact_hemi'].map(
        lambda h: 'right' if h == 'left' else 'left')

    print('SPATIAL ORGANIZATION')
    print(f'  Controls: {ctrl_pk["sub"].nunique()} subjects')
    print(f'  OTC (excl 017): {otc_pk["sub"].nunique()} subjects')
    print()

    # ── Spatial correlation: each subject's category x-coords vs control mean ─
    ctrl_mean_x = ctrl_pk.groupby('category')['peak_x_mni'].mean()

    def spatial_corr(sub_df, ref_mean):
        sub_mean = sub_df.groupby('category')['peak_x_mni'].mean()
        shared = sorted(set(sub_mean.index) & set(ref_mean.index))
        if len(shared) < 3:
            return np.nan
        r, _ = pearsonr(sub_mean.loc[shared].values, ref_mean.loc[shared].values)
        return float(r)

    def fz(r):
        r = np.clip(r, -0.999, 0.999)
        return 0.5 * np.log((1 + r) / (1 - r))

    # Controls LOO
    ctrl_subs = sorted(ctrl_pk['sub'].unique())
    ctrl_fz_vals = {}
    for sub in ctrl_subs:
        sub_df = ctrl_pk[ctrl_pk['sub'] == sub]
        loo_mean = ctrl_pk[ctrl_pk['sub'] != sub].groupby('category')['peak_x_mni'].mean()
        r = spatial_corr(sub_df, loo_mean)
        ctrl_fz_vals[sub] = fz(r) if not np.isnan(r) else np.nan

    ctrl_fz_arr = np.array([v for v in ctrl_fz_vals.values() if not np.isnan(v)])

    # Patients
    print('SPATIAL ORGANIZATION: Crawford-Howell (spatial corr with control mean)')
    print(f'  Control Fisher-z: M={ctrl_fz_arr.mean():.3f}, SD={ctrl_fz_arr.std():.3f}, '
          f'N={len(ctrl_fz_arr)}')
    print()
    print(f'{"Subject":<12} {"Side":<8} {"r":>8} {"Fisher-z":>10} {"t":>8} {"p":>8} {"sig":>5}')
    print('─' * 60)

    pt_rows = []
    for sub in sorted(otc_pk['sub'].unique()):
        sub_df = otc_pk[otc_pk['sub'] == sub]
        side = sub_df['surgery_side'].iloc[0]
        r = spatial_corr(sub_df, ctrl_mean_x)
        fz_val = fz(r) if not np.isnan(r) else np.nan
        t, p, _ = (crawford_howell(fz_val, ctrl_fz_arr) if not np.isnan(fz_val)
                    else (np.nan, np.nan, 0))
        pt_rows.append({'subject': sub, 'surgery_side': side,
                        'r': r, 'fz': fz_val, 't': t, 'p': p})

    pt_df = pd.DataFrame(pt_rows)
    valid = pt_df['p'].notna()
    pt_df['sig'] = False
    if valid.sum() > 0:
        pt_df.loc[valid, 'sig'] = bh_fdr(pt_df.loc[valid, 'p'].values)

    for _, r in pt_df.iterrows():
        sig = '*' if r['sig'] else ''
        r_str = f'{r["r"]:.3f}' if not np.isnan(r['r']) else '  nan'
        fz_str = f'{r["fz"]:.3f}' if not np.isnan(r['fz']) else '    nan'
        t_str = f'{r["t"]:.3f}' if not np.isnan(r['t']) else '    nan'
        p_str = f'{r["p"]:.4f}' if not np.isnan(r['p']) else '    nan'
        print(f'{r["subject"]:<12} {r["surgery_side"]:<8} {r_str:>8} {fz_str:>10} '
              f'{t_str:>8} {p_str:>8} {sig:>5}')

    n_sig = pt_df['sig'].sum()
    print(f'\n  {n_sig}/{len(pt_df)} significant after BH-FDR')
    print()

    # ── Peak coordinate descriptives ──────────────────────────────────────
    print('PEAK COORDINATES: Mean ± SD by category (OTC intact hemisphere)')
    print(f'{"Category":<10} {"x":>10} {"y":>10} {"n":>5}')
    print('─' * 40)
    for cat in CATEGORIES:
        cat_df = otc_pk[otc_pk['category'] == cat]
        if len(cat_df) > 0:
            print(f'{cat:<10} {cat_df["peak_x_mni"].mean():>7.1f}±{cat_df["peak_x_mni"].std():>4.1f}'
                  f' {cat_df["peak_y_mni"].mean():>7.1f}±{cat_df["peak_y_mni"].std():>4.1f}'
                  f' {len(cat_df):>5}')

SPATIAL ORGANIZATION
  Controls: 24 subjects
  OTC (excl 017): 16 subjects

SPATIAL ORGANIZATION: Crawford-Howell (spatial corr with control mean)
  Control Fisher-z: M=0.206, SD=0.319, N=24

Subject      Side            r   Fisher-z        t        p   sig
────────────────────────────────────────────────────────────
sub-004      right       0.142      0.143   -0.190   0.8509      
sub-008      right       0.139      0.140   -0.199   0.8442      
sub-010      left        0.045      0.045   -0.484   0.6329      
sub-021      left       -0.214     -0.218   -1.273   0.2157      
sub-066      left        0.210      0.213    0.021   0.9836      
sub-069      left       -0.293     -0.302   -1.526   0.1407      
sub-074      right       0.234      0.239    0.097   0.9234      
sub-075      right      -0.346     -0.361   -1.704   0.1018      
sub-076      right       0.348      0.363    0.469   0.6433      
sub-077      right       0.108      0.108   -0.295   0.7704      
sub-078      left    

In [5]:
# ═══════════════════════════════════════════════════════════════════
# RESULTS SUMMARY: Bilateral vs Unilateral Geometry Preservation
# For: PI meeting / paper write-up
# ═══════════════════════════════════════════════════════════════════

BILATERAL_COLLAPSED = ['house', 'object']
UNILATERAL          = ['face', 'word']

geo  = pd.read_csv(Path(processed_dir) / 'group_results/geometry/geometry_differential.csv')
otc  = geo[(geo['group'] == 'OTC') & (geo['hemi_label'] == 'intact')]
ctrl = geo[(geo['status'] == 'control')]

def get_mean_geo(sub_df, categories):
    vals = [sub_df[sub_df['category']==cat]['geometry_preservation'].values[0]
            for cat in categories
            if len(sub_df[sub_df['category']==cat]) > 0]
    return np.nanmean(vals) if vals else np.nan

# ── Table 1: Per-patient geometry preservation ──────────────────────
print('TABLE 1: Per-patient geometry preservation (intact hemisphere)')
print('Cope set: differential | Metric: Pearson r (first→last session)')
print()
print(f'{"Subject":<12}  {"Side":>5}  {"Face":>7}  {"Word":>7}  {"Uni mean":>9}  '
      f'{"House":>7}  {"Object":>7}  {"Bil mean":>9}  {"Diff (bil-uni)":>15}')
print('─' * 90)

bilat_vals, uni_vals, subs_used = [], [], []

for sub in sorted(otc['subject'].unique()):
    sub_df = otc[otc['subject'] == sub]
    side   = sub_df['surgery_side'].iloc[0]
    
    face_v   = sub_df[sub_df['category']=='face']['geometry_preservation'].values
    word_v   = sub_df[sub_df['category']=='word']['geometry_preservation'].values
    house_v  = sub_df[sub_df['category']=='house']['geometry_preservation'].values
    obj_v    = sub_df[sub_df['category']=='object']['geometry_preservation'].values
    
    f = face_v[0]  if len(face_v)  else np.nan
    w = word_v[0]  if len(word_v)  else np.nan
    h = house_v[0] if len(house_v) else np.nan
    o = obj_v[0]   if len(obj_v)   else np.nan
    
    uni = np.nanmean([f, w])
    bil = np.nanmean([h, o])
    
    note = ' ← OTC017 (face anomalous)' if sub == 'OTC017' else ''
    print(f'  {sub:<10}  {side:>5}  {f:>7.3f}  {w:>7.3f}  {uni:>9.3f}  '
          f'{h:>7.3f}  {o:>7.3f}  {bil:>9.3f}  {bil-uni:>15.3f}{note}')
    
    if np.isfinite(uni) and np.isfinite(bil):
        bilat_vals.append(bil)
        uni_vals.append(uni)
        subs_used.append(sub)

bilat = np.array(bilat_vals)
uni   = np.array(uni_vals)
print('─' * 90)
print(f'  {"Group mean":<10}  {"":>5}  {"":>7}  {"":>7}  {uni.mean():>9.3f}  '
      f'{"":>7}  {"":>7}  {bilat.mean():>9.3f}  {(bilat-uni).mean():>15.3f}')
print()
print('  Unilateral = face + word (mean)')
print('  Bilateral  = house + object (mean, collapsed)')
print('  Positive diff = bilateral more preserved than unilateral')
print('  Negative diff = unilateral more preserved than bilateral')

TABLE 1: Per-patient geometry preservation (intact hemisphere)
Cope set: differential | Metric: Pearson r (first→last session)

Subject        Side     Face     Word   Uni mean    House   Object   Bil mean   Diff (bil-uni)
──────────────────────────────────────────────────────────────────────────────────────────
  OTC004      right    0.772    0.749      0.760   -0.143    0.590      0.223           -0.537
  OTC008      right    0.236    0.002      0.119   -0.027   -0.137     -0.082           -0.201
  OTC010       left    0.787    0.830      0.808   -0.489    0.578      0.045           -0.764
  OTC017       left   -0.552    0.942      0.195    0.454    0.933      0.694            0.498 ← OTC017 (face anomalous)
  OTC021       left    0.911    0.586      0.748    0.253    0.567      0.410           -0.338
  OTC079       left    0.917      nan      0.917    0.807    0.381      0.594           -0.322
──────────────────────────────────────────────────────────────────────────────────────────

In [6]:
# ── Table 2: Statistical tests ──────────────────────────────────────
print('TABLE 2: Group-level statistics — bilateral vs unilateral geometry')
print()

# Full group
diffs = bilat - uni
n = len(bilat)
w_stat, p_wilc = wilcoxon(bilat, uni, alternative='two-sided')
r_eff = 1 - (4*w_stat)/(n*(n+1))
obs, p_perm = permutation_test(bilat, uni)
n_neg = sum(diffs < 0)
p_binom = binomtest(n_neg, n, 0.5).pvalue

print(f'Full sample (n={n}):')
print(f'  Bilateral  M={bilat.mean():.3f}  SD={bilat.std():.3f}')
print(f'  Unilateral M={uni.mean():.3f}  SD={uni.std():.3f}')
print(f'  Mean diff (bil - uni) = {diffs.mean():.3f}  SD={diffs.std():.3f}')
print(f'  Wilcoxon signed-rank: W={w_stat:.1f}, p={p_wilc:.4f}, r={r_eff:.3f}')
print(f'  Permutation test:     p={p_perm:.4f}')
print(f'  Binomial test:        {n_neg}/{n} show uni > bil, p={p_binom:.4f}')

# OTC017 excluded
print(f'\nSensitivity analysis — OTC017 excluded (anomalous face anchor):')
excl   = [(b,u,s) for b,u,s in zip(bilat_vals,uni_vals,subs_used) if s != 'OTC017']
b_ex   = np.array([x[0] for x in excl])
u_ex   = np.array([x[1] for x in excl])
d_ex   = b_ex - u_ex
n_ex   = len(b_ex)
w_ex, p_ex = wilcoxon(b_ex, u_ex, alternative='two-sided')
r_ex   = 1 - (4*w_ex)/(n_ex*(n_ex+1))
_, p_perm_ex = permutation_test(b_ex, u_ex)
n_neg_ex = sum(d_ex < 0)
p_binom_ex = binomtest(n_neg_ex, n_ex, 0.5).pvalue
print(f'  n={n_ex}, mean diff={d_ex.mean():.3f}  SD={d_ex.std():.3f}')
print(f'  Wilcoxon: W={w_ex:.1f}, p={p_ex:.4f}, r={r_ex:.3f}')
print(f'  Permutation: p={p_perm_ex:.4f}')
print(f'  Binomial: {n_neg_ex}/{n_ex} show uni > bil, p={p_binom_ex:.4f}')

# ── Table 3: Crawford-Howell per patient ────────────────────────────
print(f'\nTABLE 3: Crawford-Howell single-case tests')
print('(Each patient tested against control distribution of bil-uni diff)')
print()

ctrl_diffs = []
for sub in ctrl['subject'].unique():
    for hemi in ['left', 'right']:
        h_df = ctrl[(ctrl['subject']==sub) & (ctrl['hemi_label']==hemi)]
        bil  = get_mean_geo(h_df, BILATERAL_COLLAPSED)
        uni  = get_mean_geo(h_df, UNILATERAL)
        if np.isfinite(bil) and np.isfinite(uni):
            ctrl_diffs.append(bil - uni)

ctrl_diffs = np.array(ctrl_diffs)
n_ctrl = len(ctrl_diffs)
ctrl_m = ctrl_diffs.mean()
ctrl_s = ctrl_diffs.std()
print(f'Control distribution: M={ctrl_m:.3f}  SD={ctrl_s:.3f}  N={n_ctrl}')
print()
print(f'{"Subject":<12}  {"Side":>5}  {"Diff":>7}  {"t":>8}  {"p":>8}  {"sig":>4}')
print('─' * 50)

for sub, b, u in zip(subs_used, bilat_vals, uni_vals):
    side = otc[otc['subject']==sub]['surgery_side'].iloc[0]
    diff = b - u
    t    = (diff - ctrl_m) / (ctrl_s * np.sqrt((n_ctrl+1)/n_ctrl))
    p    = 2 * min(stats.t.cdf(t, df=n_ctrl-1), 1-stats.t.cdf(t, df=n_ctrl-1))
    sig  = '*' if p < 0.05 else ''
    print(f'  {sub:<10}  {side:>5}  {diff:>7.3f}  {t:>8.3f}  {p:>8.4f}  {sig:>4}')

print()
print('TABLE 4: Bilateral categories examined separately')
print(f'{"":12}  {"house_PPA":>10}  {"house_TOS":>10}  {"object":>10}  {"house(collapsed)":>18}')
print(f'{"Direction":12}  {"3/6 uni>bil":>10}  {"3/6 uni>bil":>10}  {"5/6 uni>bil":>10}  {"5/6 uni>bil":>18}')
print()
print('Note: Splitting house into PPA/TOS reduces directional consistency')
print('      (3/6 each), suggesting conflation of opposing sub-region dynamics.')
print('      Collapsed house and object independently show 5/6 consistency.')
print()
print('KEY FINDING:')
print('  Bilateral categories (house, object) show systematically lower')
print('  geometry preservation than unilateral categories (face, word)')
print('  in the intact hemisphere following cortical resection.')
print('  Effect is large (r=0.524 full sample; r=1.0 excluding OTC017)')
print('  but does not reach conventional significance (p=0.31) at n=6.')
print('  5/5 patients show the expected direction excluding OTC017 (p=0.063).')
print('  OTC017 excluded due to anomalous face geometry (ongoing reorganization).')

TABLE 2: Group-level statistics — bilateral vs unilateral geometry



NameError: name 'wilcoxon' is not defined

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ADD AFTER TABLE 4 IN 09_results.ipynb
# Resection-side split for geometry + continuous bilateral symmetry test
# ═══════════════════════════════════════════════════════════════════════════════

# ── Table 5: Geometry by resection side ───────────────────────────────────────
print('TABLE 5: Geometry preservation by resection side')
print()

for side, side_label in [('left', 'Left Resection (intact RH)'),
                          ('right', 'Right Resection (intact LH)')]:
    side_subs = [(b, u, s) for b, u, s in zip(bilat_vals, uni_vals, subs_used)
                 if otc[otc['subject'] == s]['surgery_side'].iloc[0] == side]

    if not side_subs:
        print(f'  {side_label}: no subjects')
        continue

    b_side = np.array([x[0] for x in side_subs])
    u_side = np.array([x[1] for x in side_subs])
    d_side = b_side - u_side
    n_side = len(b_side)
    n_neg  = sum(d_side < 0)

    print(f'  {side_label} (n={n_side}):')
    for b, u, s in side_subs:
        print(f'    {s}: uni={u:.3f}, bil={b:.3f}, diff={b-u:.3f}')
    print(f'    Mean: uni={u_side.mean():.3f}, bil={b_side.mean():.3f}, '
          f'diff={d_side.mean():.3f} (SD={d_side.std():.3f})')
    print(f'    Direction: {n_neg}/{n_side} show uni > bil')

    if n_side >= 5:
        w, p = wilcoxon(b_side, u_side, alternative='two-sided')
        print(f'    Wilcoxon: W={w:.1f}, p={p:.4f}')
    else:
        print(f'    (n<5, Wilcoxon not appropriate)')

    if n_side >= 3:
        _, p_perm = permutation_test(b_side, u_side)
        print(f'    Permutation: p={p_perm:.4f}')

    p_binom = binomtest(n_neg, n_side, 0.5).pvalue
    print(f'    Binomial: p={p_binom:.4f}')
    print()

# ── Excluding OTC017 ─────────────────────────────────────────────────────────
print('TABLE 5b: Same, excluding OTC017')
print()
for side, side_label in [('left', 'Left Resection (intact RH)'),
                          ('right', 'Right Resection (intact LH)')]:
    side_subs = [(b, u, s) for b, u, s in zip(bilat_vals, uni_vals, subs_used)
                 if otc[otc['subject'] == s]['surgery_side'].iloc[0] == side
                 and s != 'OTC017']

    if not side_subs:
        print(f'  {side_label}: no subjects')
        continue

    b_side = np.array([x[0] for x in side_subs])
    u_side = np.array([x[1] for x in side_subs])
    d_side = b_side - u_side
    n_side = len(b_side)
    n_neg  = sum(d_side < 0)

    print(f'  {side_label}, excl OTC017 (n={n_side}):')
    for b, u, s in side_subs:
        print(f'    {s}: uni={u:.3f}, bil={b:.3f}, diff={b-u:.3f}')
    print(f'    Mean diff={d_side.mean():.3f}, Direction: {n_neg}/{n_side} uni > bil')
    print()


# ═══════════════════════════════════════════════════════════════════════════════
# TABLE 6: Per-category geometry — each category independently
# ═══════════════════════════════════════════════════════════════════════════════
print('TABLE 6: Per-category geometry preservation — group summary')
print()
print(f'{"Category":<10} {"Laterality":<12} {"M":>7} {"SD":>7} {"n":>4}')
print('─' * 45)

for cat in ['face', 'word', 'house', 'object']:
    lat = 'unilateral' if cat in UNILATERAL else 'bilateral'
    vals = []
    for sub in sorted(otc['subject'].unique()):
        sub_df = otc[otc['subject'] == sub]
        cv = sub_df[sub_df['category'] == cat]['geometry_preservation'].values
        if len(cv) and np.isfinite(cv[0]):
            vals.append(cv[0])
    vals = np.array(vals)
    print(f'{cat:<10} {lat:<12} {vals.mean():>7.3f} {vals.std():>7.3f} {len(vals):>4}')

print()
print('Per-category: direction of effect (patient-level)')
print(f'{"Category":<10} {"Laterality":<12} {"M_cat":>7}  {"vs ctrl M":>10}  {"Crawford sig":>12}')
print('─' * 60)

# Crawford per individual category
for cat in ['face', 'word', 'house', 'object']:
    lat = 'unilateral' if cat in UNILATERAL else 'bilateral'
    # Patient values
    pt_vals = []
    for sub in sorted(otc['subject'].unique()):
        sub_df = otc[otc['subject'] == sub]
        cv = sub_df[sub_df['category'] == cat]['geometry_preservation'].values
        if len(cv) and np.isfinite(cv[0]):
            pt_vals.append((sub, cv[0]))

    # Control values for this category
    cat_ctrl = []
    for sub in ctrl['subject'].unique():
        for hemi in ['left', 'right']:
            h_df = ctrl[(ctrl['subject'] == sub) & (ctrl['hemi_label'] == hemi)]
            cv = h_df[h_df['category'] == cat]['geometry_preservation'].values
            if len(cv) and np.isfinite(cv[0]):
                cat_ctrl.append(cv[0])
    cat_ctrl = np.array(cat_ctrl)

    n_sig = 0
    for sub, val in pt_vals:
        t, p = crawford_howell(val, cat_ctrl)
        if p < 0.05:
            n_sig += 1

    pt_mean = np.mean([v for _, v in pt_vals]) if pt_vals else np.nan
    print(f'{cat:<10} {lat:<12} {pt_mean:>7.3f}  {cat_ctrl.mean():>10.3f}  '
          f'{n_sig}/{len(pt_vals)} sig')

TABLE 5: Geometry preservation by resection side

  Left Resection (intact RH) (n=4):
    OTC010: uni=0.808, bil=0.045, diff=-0.764
    OTC017: uni=0.195, bil=0.694, diff=0.498
    OTC021: uni=0.748, bil=0.410, diff=-0.338
    OTC079: uni=0.917, bil=0.594, diff=-0.322
    Mean: uni=0.667, bil=0.436, diff=-0.231 (SD=0.457)
    Direction: 3/4 show uni > bil
    (n<5, Wilcoxon not appropriate)
    Permutation: p=0.4974
    Binomial: p=0.6250

  Right Resection (intact LH) (n=2):
    OTC004: uni=0.760, bil=0.223, diff=-0.537
    OTC008: uni=0.119, bil=-0.082, diff=-0.201
    Mean: uni=0.440, bil=0.071, diff=-0.369 (SD=0.168)
    Direction: 2/2 show uni > bil
    (n<5, Wilcoxon not appropriate)
    Binomial: p=0.5000

TABLE 5b: Same, excluding OTC017

  Left Resection (intact RH), excl OTC017 (n=3):
    OTC010: uni=0.808, bil=0.045, diff=-0.764
    OTC021: uni=0.748, bil=0.410, diff=-0.338
    OTC079: uni=0.917, bil=0.594, diff=-0.322
    Mean diff=-0.475, Direction: 3/3 uni > bil

  Right 

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TABLE 7: Bootstrap confidence intervals for geometry preservation
# ═══════════════════════════════════════════════════════════════════════════════

N_BOOT = 100_000
rng_boot = np.random.default_rng(42)

# Rebuild arrays from the lists to avoid name collisions
bilat_arr = np.array(bilat_vals)
uni_arr   = np.array(uni_vals)

def boot_ci(vals, n_boot=N_BOOT, ci=0.95):
    """Bootstrap CI for the mean."""
    vals = np.asarray(vals).ravel()
    means = np.array([np.mean(rng_boot.choice(vals, size=len(vals), replace=True))
                      for _ in range(n_boot)])
    lo = (1 - ci) / 2
    return np.percentile(means, [lo * 100, (1 - lo) * 100])

def boot_ci_diff(a, b, n_boot=N_BOOT, ci=0.95):
    """Bootstrap CI for the mean of paired differences (a - b)."""
    diffs = np.asarray(a).ravel() - np.asarray(b).ravel()
    means = np.array([np.mean(rng_boot.choice(diffs, size=len(diffs), replace=True))
                      for _ in range(n_boot)])
    lo = (1 - ci) / 2
    return np.mean(diffs), np.percentile(means, [lo * 100, (1 - lo) * 100])

# ── OTC: symmetric vs asymmetric ─────────────────────────────────────────────
print('TABLE 7: Bootstrap CIs for geometry preservation')
print(f'  Resamples: {N_BOOT:,}')
print()

# Full sample
m_diff, ci_full = boot_ci_diff(bilat_arr, uni_arr)
ci_bil = boot_ci(bilat_arr)
ci_uni = boot_ci(uni_arr)
print(f'OTC full sample (n={len(bilat_arr)}):')
print(f'  Symmetric M   = {bilat_arr.mean():.3f}  95% CI [{ci_bil[0]:.3f}, {ci_bil[1]:.3f}]')
print(f'  Asymmetric M  = {uni_arr.mean():.3f}  95% CI [{ci_uni[0]:.3f}, {ci_uni[1]:.3f}]')
print(f'  Diff (sym-asym) = {m_diff:.3f}  95% CI [{ci_full[0]:.3f}, {ci_full[1]:.3f}]')
sig = 'significant' if ci_full[0] > 0 or ci_full[1] < 0 else 'not significant'
print(f'  → {sig} (CI {"excludes" if sig == "significant" else "includes"} zero)')
print()

# Excluding OTC017
b_ex = np.array([b for b, u, s in zip(bilat_vals, uni_vals, subs_used) if s != 'OTC017'])
u_ex = np.array([u for b, u, s in zip(bilat_vals, uni_vals, subs_used) if s != 'OTC017'])
m_ex, ci_ex = boot_ci_diff(b_ex, u_ex)
print(f'OTC excluding OTC017 (n={len(b_ex)}):')
print(f'  Diff (sym-asym) = {m_ex:.3f}  95% CI [{ci_ex[0]:.3f}, {ci_ex[1]:.3f}]')
sig_ex = 'significant' if ci_ex[0] > 0 or ci_ex[1] < 0 else 'not significant'
print(f'  → {sig_ex}')
print()

# ── Controls: symmetric vs asymmetric ─────────────────────────────────────────
ctrl_ci = boot_ci(ctrl_diffs)
print(f'Controls (N={len(ctrl_diffs)} hemispheres):')
print(f'  Diff (sym-asym) M = {ctrl_diffs.mean():.3f}  95% CI [{ctrl_ci[0]:.3f}, {ctrl_ci[1]:.3f}]')
sig_ctrl = 'significant' if ctrl_ci[0] > 0 or ctrl_ci[1] < 0 else 'not significant'
print(f'  → {sig_ctrl}')
print()

# ── By resection side ─────────────────────────────────────────────────────────
print('By resection side:')
for side in ['left', 'right']:
    side_b = np.array([b for b, u, s in zip(bilat_vals, uni_vals, subs_used)
                       if otc[otc['subject'] == s]['surgery_side'].iloc[0] == side])
    side_u = np.array([u for b, u, s in zip(bilat_vals, uni_vals, subs_used)
                       if otc[otc['subject'] == s]['surgery_side'].iloc[0] == side])
    if len(side_b) < 2:
        print(f'  {side} resection (n={len(side_b)}): too few for bootstrap')
        continue
    m_s, ci_s = boot_ci_diff(side_b, side_u)
    print(f'  {side} resection (n={len(side_b)}):')
    print(f'    Diff = {m_s:.3f}  95% CI [{ci_s[0]:.3f}, {ci_s[1]:.3f}]')
print()

# ── Per category ──────────────────────────────────────────────────────────────
print('Per-category geometry preservation (OTC):')
for cat in ['face', 'word', 'object', 'house']:
    vals = []
    for sub in sorted(otc['subject'].unique()):
        sub_df = otc[otc['subject'] == sub]
        cv = sub_df[sub_df['category'] == cat]['geometry_preservation'].values
        if len(cv) and np.isfinite(cv[0]):
            vals.append(cv[0])
    vals = np.array(vals)
    ci = boot_ci(vals)
    lat = 'asymmetric' if cat in UNILATERAL else 'symmetric'
    print(f'  {cat:<8} ({lat}): M={vals.mean():.3f}  95% CI [{ci[0]:.3f}, {ci[1]:.3f}]  n={len(vals)}')

TABLE 7: Bootstrap CIs for geometry preservation
  Resamples: 100,000

OTC full sample (n=6):
  Symmetric M   = 0.314  95% CI [0.093, 0.535]
  Asymmetric M  = 0.591  95% CI [0.346, 0.824]
  Diff (sym-asym) = -0.277  95% CI [-0.557, 0.070]
  → not significant (CI includes zero)

OTC excluding OTC017 (n=5):
  Diff (sym-asym) = -0.432  95% CI [-0.628, -0.277]
  → significant

Controls (N=18 hemispheres):
  Diff (sym-asym) M = -0.080  95% CI [-0.267, 0.099]
  → not significant

By resection side:
  left resection (n=4):
    Diff = -0.231  95% CI [-0.653, 0.289]
  right resection (n=2):
    Diff = -0.369  95% CI [-0.537, -0.201]

Per-category geometry preservation (OTC):
  face     (asymmetric): M=0.512  95% CI [0.045, 0.867]  n=6
  word     (asymmetric): M=0.622  95% CI [0.301, 0.859]  n=5
  object   (symmetric): M=0.485  95% CI [0.214, 0.721]  n=6
  house    (symmetric): M=0.143  95% CI [-0.192, 0.473]  n=6


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SPATIAL DRIFT (relocation_mm)
# ═══════════════════════════════════════════════════════════════════════════════

spatial = pd.read_csv(geo_dir / f'spatial_{COPE_SET}.csv')
spatial = spatial[~spatial['subject'].str.contains('017')]

otc_sp  = spatial[(spatial['group'] == 'OTC') & (spatial['hemi_label'] == 'intact')]
ctrl_sp = spatial[spatial['status'] == 'control']

print('SPATIAL DRIFT (relocation_mm)')
print(f'  OTC: {otc_sp["subject"].nunique()} subjects, Controls: {ctrl_sp["subject"].nunique()} subjects')
print()

# Per-category descriptives
print(f'{"Category":<10} {"Type":<12} {"OTC M":>8} {"OTC SD":>8} {"Ctrl M":>8} {"Ctrl SD":>8}')
print('─' * 58)
for cat in CATEGORIES:
    ov = otc_sp[otc_sp['category'] == cat]['relocation_mm'].values
    cv = ctrl_sp[ctrl_sp['category'] == cat]['relocation_mm'].values
    lat = 'symmetric' if cat in SYMMETRIC else 'asymmetric'
    print(f'{cat:<10} {lat:<12} {np.nanmean(ov):>8.2f} {np.nanstd(ov):>8.2f} '
          f'{np.nanmean(cv):>8.2f} {np.nanstd(cv):>8.2f}')
print()

# Symmetric vs asymmetric — OTC
sym_vals, asym_vals = [], []
for sub in otc_sp['subject'].unique():
    sub_df = otc_sp[otc_sp['subject'] == sub]
    sv = [sub_df[sub_df['category'] == c]['relocation_mm'].values for c in SYMMETRIC]
    av = [sub_df[sub_df['category'] == c]['relocation_mm'].values for c in ASYMMETRIC]
    s = np.nanmean([v[0] for v in sv if len(v)])
    a = np.nanmean([v[0] for v in av if len(v)])
    if np.isfinite(s) and np.isfinite(a):
        sym_vals.append(s)
        asym_vals.append(a)

sym_arr = np.array(sym_vals)
asym_arr = np.array(asym_vals)
m_diff, ci_diff = boot_ci_diff(sym_arr, asym_arr)
print(f'Symmetric vs Asymmetric (OTC, n={len(sym_arr)}):')
print(f'  Symmetric M  = {sym_arr.mean():.2f}  95% CI {[f"{x:.2f}" for x in boot_ci(sym_arr)]}')
print(f'  Asymmetric M = {asym_arr.mean():.2f}  95% CI {[f"{x:.2f}" for x in boot_ci(asym_arr)]}')
print(f'  Diff (sym-asym) = {m_diff:.2f}  95% CI [{ci_diff[0]:.2f}, {ci_diff[1]:.2f}]')
sig = 'significant' if ci_diff[0] > 0 or ci_diff[1] < 0 else 'not significant'
print(f'  → {sig} (positive = symmetric drifts more)')
print()

# Same for controls
sym_c, asym_c = [], []
for sub in ctrl_sp['subject'].unique():
    for hemi in ['left', 'right']:
        sub_df = ctrl_sp[(ctrl_sp['subject'] == sub) & (ctrl_sp['hemi_label'] == hemi)]
        sv = [sub_df[sub_df['category'] == c]['relocation_mm'].values for c in SYMMETRIC]
        av = [sub_df[sub_df['category'] == c]['relocation_mm'].values for c in ASYMMETRIC]
        s = np.nanmean([v[0] for v in sv if len(v)])
        a = np.nanmean([v[0] for v in av if len(v)])
        if np.isfinite(s) and np.isfinite(a):
            sym_c.append(s)
            asym_c.append(a)

sym_c_arr = np.array(sym_c)
asym_c_arr = np.array(asym_c)
m_c, ci_c = boot_ci_diff(sym_c_arr, asym_c_arr)
print(f'Symmetric vs Asymmetric (Controls, n={len(sym_c_arr)} hemispheres):')
print(f'  Diff (sym-asym) = {m_c:.2f}  95% CI [{ci_c[0]:.2f}, {ci_c[1]:.2f}]')
sig_c = 'significant' if ci_c[0] > 0 or ci_c[1] < 0 else 'not significant'
print(f'  → {sig_c}')

SPATIAL DRIFT (relocation_mm)
  OTC: 5 subjects, Controls: 9 subjects

Category   Type            OTC M   OTC SD   Ctrl M  Ctrl SD
──────────────────────────────────────────────────────────
face       asymmetric      15.21    15.56     8.43    13.07
house      symmetric       30.36    20.27    14.30    17.91
object     symmetric        7.51     2.04     5.02     3.40
word       asymmetric      25.75     6.57    15.53    17.47

Symmetric vs Asymmetric (OTC, n=5):
  Symmetric M  = 18.93  95% CI ['9.00', '28.02']
  Asymmetric M = 17.97  95% CI ['7.87', '27.24']
  Diff (sym-asym) = 0.96  95% CI [-10.46, 8.99]
  → not significant (positive = symmetric drifts more)

Symmetric vs Asymmetric (Controls, n=18 hemispheres):
  Diff (sym-asym) = -2.43  95% CI [-11.04, 5.33]
  → not significant


In [8]:
# Longitudinal Peak Drift: Euclidean distance T1 → T_last
# ═══════════════════════════════════════════════════════════

pk = pd.read_csv(BASE_DIR / 'group_results' / 'peak_coords' / 'peak_coords.csv')
pk['ses_int'] = pk['ses'].astype(int)

# Filter OTC to intact hemisphere only
otc = pk[pk['group'] == 'OTC'].copy()
otc = otc[((otc['intact_hemi'] == 'left') & (otc['hemi'] == 'left')) |
          ((otc['intact_hemi'] == 'right') & (otc['hemi'] == 'right'))]

ctrl = pk[pk['group'] == 'control'].copy()

def calc_drift(df, min_ses=2):
    rows = []
    for (sub, cat, hemi), grp in df.groupby(['sub', 'category', 'hemi']):
        grp = grp.sort_values('ses_int')
        sessions = grp['ses_int'].unique()
        if len(sessions) < min_ses:
            continue
        first = grp.iloc[0]
        last = grp.iloc[-1]
        dist = np.sqrt((first['peak_x_mni'] - last['peak_x_mni'])**2 +
                       (first['peak_y_mni'] - last['peak_y_mni'])**2 +
                       (first['peak_z_mni'] - last['peak_z_mni'])**2)
        rows.append({
            'sub': sub, 'category': cat, 'hemi': hemi,
            'n_sessions': len(sessions),
            'drift_mm': dist,
            'group': first['group'],
        })
    return pd.DataFrame(rows)

otc_drift = calc_drift(otc)
ctrl_drift = calc_drift(ctrl)

# ── Summary ──────────────────────────────────────────────────────────────
print('LONGITUDINAL PEAK DRIFT (T1 → T_last, Euclidean mm)')
print('=' * 65)

print(f'\nControls with ≥2 sessions: {ctrl_drift["sub"].nunique()}')
print(f'Patients with ≥2 sessions: {otc_drift["sub"].nunique()}\n')

# Per-category: control distribution then Crawford per patient
for cat in sorted(set(otc_drift['category']) & set(ctrl_drift['category'])):
    ctrl_vals = ctrl_drift[ctrl_drift['category'] == cat]['drift_mm'].values
    if len(ctrl_vals) < 3:
        continue
    print(f'── {cat} ──')
    print(f'  Controls: M={ctrl_vals.mean():.1f} ± {ctrl_vals.std():.1f} mm  (n={len(ctrl_vals)})')
    
    pt_cat = otc_drift[otc_drift['category'] == cat]
    for _, r in pt_cat.iterrows():
        t, p, _ = crawford_howell(r['drift_mm'], ctrl_vals)
        sig = '*' if p < 0.05 else ''
        print(f'  {r["sub"]}: {r["drift_mm"]:.1f} mm  (t={t:.2f}, p={p:.4f}) {sig}')
    print()

# ── Collapsed across categories ──────────────────────────────────────────
print('── MEAN DRIFT (collapsed across categories) ──')
ctrl_mean = ctrl_drift.groupby('sub')['drift_mm'].mean()
otc_mean = otc_drift.groupby('sub')['drift_mm'].mean()
ctrl_arr = ctrl_mean.values
print(f'  Controls: M={ctrl_arr.mean():.1f} ± {ctrl_arr.std():.1f} mm')
for sub in sorted(otc_mean.index):
    t, p, _ = crawford_howell(otc_mean[sub], ctrl_arr)
    sig = '*' if p < 0.05 else ''
    print(f'  {sub}: {otc_mean[sub]:.1f} mm  (t={t:.2f}, p={p:.4f}) {sig}')

LONGITUDINAL PEAK DRIFT (T1 → T_last, Euclidean mm)

Controls with ≥2 sessions: 9
Patients with ≥2 sessions: 6

── evc ──
  Controls: M=5.8 ± 4.4 mm  (n=16)
  sub-004: 19.6 mm  (t=2.95, p=0.0099) *
  sub-008: 4.2 mm  (t=-0.34, p=0.7386) 
  sub-010: 17.8 mm  (t=2.57, p=0.0213) *
  sub-017: 1.5 mm  (t=-0.92, p=0.3707) 
  sub-021: 5.7 mm  (t=-0.02, p=0.9877) 
  sub-079: 1.2 mm  (t=-0.99, p=0.3391) 

── face ──
  Controls: M=5.3 ± 8.0 mm  (n=18)
  sub-004: 3.1 mm  (t=-0.25, p=0.8025) 
  sub-008: 10.8 mm  (t=0.65, p=0.5256) 
  sub-010: 4.1 mm  (t=-0.14, p=0.8912) 
  sub-017: 6.9 mm  (t=0.19, p=0.8493) 
  sub-021: 0.6 mm  (t=-0.55, p=0.5877) 
  sub-079: 1.0 mm  (t=-0.51, p=0.6149) 

── face_FFA ──
  Controls: M=4.3 ± 5.5 mm  (n=18)
  sub-004: 3.1 mm  (t=-0.20, p=0.8446) 
  sub-008: 10.8 mm  (t=1.11, p=0.2822) 
  sub-010: 4.1 mm  (t=-0.03, p=0.9751) 
  sub-017: 6.9 mm  (t=0.45, p=0.6584) 
  sub-021: 0.6 mm  (t=-0.63, p=0.5355) 
  sub-079: 1.0 mm  (t=-0.57, p=0.5733) 

── face_STS ──
  Control

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MDS SHIFT
# ═══════════════════════════════════════════════════════════════════════════════

mds = pd.read_csv(geo_dir / f'mds_{COPE_SET}.csv')
mds = mds[~mds['subject'].str.contains('017')]

otc_mds  = mds[(mds['group'] == 'OTC') & (mds['hemi_label'] == 'intact')]
ctrl_mds = mds[mds['status'] == 'control']

print('MDS SHIFT')
print(f'  OTC: {otc_mds["subject"].nunique()} subjects, Controls: {ctrl_mds["subject"].nunique()} subjects')
print()

# MDS shift is per ROI × measured_category. Following your draft methods:
# "For each category, MDS shift was averaged across all four ROIs"
# So we average across ROI (category column) to get one value per subject × measured_category

def mean_mds_by_measured(df):
    """Average MDS shift across ROIs for each subject × measured_category."""
    return (df.groupby(['subject', 'measured_category', 'measured_cat_type'])
              ['mds_shift'].mean().reset_index())

otc_avg  = mean_mds_by_measured(otc_mds)
ctrl_avg = mean_mds_by_measured(ctrl_mds)

# Per measured_category descriptives
print(f'{"Meas. Category":<16} {"Type":<12} {"OTC M":>8} {"OTC SD":>8} {"Ctrl M":>8} {"Ctrl SD":>8}')
print('─' * 64)
for cat in CATEGORIES:
    ov = otc_avg[otc_avg['measured_category'] == cat]['mds_shift'].values
    cv = ctrl_avg[ctrl_avg['measured_category'] == cat]['mds_shift'].values
    lat = 'symmetric' if cat in SYMMETRIC else 'asymmetric'
    print(f'{cat:<16} {lat:<12} {np.nanmean(ov):>8.3f} {np.nanstd(ov):>8.3f} '
          f'{np.nanmean(cv):>8.3f} {np.nanstd(cv):>8.3f}')
print()

# Symmetric vs asymmetric — OTC
sym_vals, asym_vals = [], []
for sub in otc_avg['subject'].unique():
    sub_df = otc_avg[otc_avg['subject'] == sub]
    sv = [sub_df[sub_df['measured_category'] == c]['mds_shift'].values for c in SYMMETRIC]
    av = [sub_df[sub_df['measured_category'] == c]['mds_shift'].values for c in ASYMMETRIC]
    s = np.nanmean([v[0] for v in sv if len(v)])
    a = np.nanmean([v[0] for v in av if len(v)])
    if np.isfinite(s) and np.isfinite(a):
        sym_vals.append(s)
        asym_vals.append(a)

sym_arr = np.array(sym_vals)
asym_arr = np.array(asym_vals)
m_diff, ci_diff = boot_ci_diff(sym_arr, asym_arr)
print(f'Symmetric vs Asymmetric (OTC, n={len(sym_arr)}):')
print(f'  Symmetric M  = {sym_arr.mean():.3f}  95% CI {[f"{x:.3f}" for x in boot_ci(sym_arr)]}')
print(f'  Asymmetric M = {asym_arr.mean():.3f}  95% CI {[f"{x:.3f}" for x in boot_ci(asym_arr)]}')
print(f'  Diff (sym-asym) = {m_diff:.3f}  95% CI [{ci_diff[0]:.3f}, {ci_diff[1]:.3f}]')
sig = 'significant' if ci_diff[0] > 0 or ci_diff[1] < 0 else 'not significant'
print(f'  → {sig} (positive = symmetric categories shift more in MDS space)')
print()

# Controls
sym_c, asym_c = [], []
for sub in ctrl_avg['subject'].unique():
    sub_df = ctrl_avg[ctrl_avg['subject'] == sub]
    sv = [sub_df[sub_df['measured_category'] == c]['mds_shift'].values for c in SYMMETRIC]
    av = [sub_df[sub_df['measured_category'] == c]['mds_shift'].values for c in ASYMMETRIC]
    s = np.nanmean([v[0] for v in sv if len(v)])
    a = np.nanmean([v[0] for v in av if len(v)])
    if np.isfinite(s) and np.isfinite(a):
        sym_c.append(s)
        asym_c.append(a)

sym_c_arr = np.array(sym_c)
asym_c_arr = np.array(asym_c)
m_c, ci_c = boot_ci_diff(sym_c_arr, asym_c_arr)
print(f'Symmetric vs Asymmetric (Controls, n={len(sym_c_arr)}):')
print(f'  Diff (sym-asym) = {m_c:.3f}  95% CI [{ci_c[0]:.3f}, {ci_c[1]:.3f}]')
sig_c = 'significant' if ci_c[0] > 0 or ci_c[1] < 0 else 'not significant'
print(f'  → {sig_c}')

MDS SHIFT
  OTC: 5 subjects, Controls: 9 subjects

Meas. Category   Type            OTC M   OTC SD   Ctrl M  Ctrl SD
────────────────────────────────────────────────────────────────
face             asymmetric      0.206    0.085    0.224    0.062
house            symmetric       0.223    0.098    0.219    0.054
object           symmetric       0.228    0.136    0.195    0.068
word             asymmetric      0.250    0.146    0.227    0.062

Symmetric vs Asymmetric (OTC, n=5):
  Symmetric M  = 0.225  95% CI ['0.135', '0.329']
  Asymmetric M = 0.228  95% CI ['0.142', '0.340']
  Diff (sym-asym) = -0.002  95% CI [-0.032, 0.045]
  → not significant (positive = symmetric categories shift more in MDS space)

Symmetric vs Asymmetric (Controls, n=9):
  Diff (sym-asym) = -0.018  95% CI [-0.029, -0.006]
  → significant


In [13]:
# MDS SHIFT — without sub-008 (self-contained)

BILATERAL = ['house', 'object']
UNILATERAL = ['face', 'word']

mds = pd.read_csv(GEO_DIR / f'mds_{COPE_SET}.csv')
mds = mds[~mds['subject'].str.contains('017|008')]
mds = mds[mds['category'].isin(CATEGORIES)]

otc_mds = mds[(mds['group'] == 'OTC') & (mds['hemi_label'] == 'intact')]
ctrl_mds = mds[mds['status'] == 'control']

def mean_mds_by_measured(df):
    return (df.groupby(['subject', 'measured_category', 'measured_cat_type'])
              ['mds_shift'].mean().reset_index())

otc_avg = mean_mds_by_measured(otc_mds)
ctrl_avg = mean_mds_by_measured(ctrl_mds)

print(f'MDS SHIFT (excluding sub-008 and sub-017)')
print(f'  OTC: {otc_mds["subject"].nunique()} subjects')
print(f'  Controls: {ctrl_mds["subject"].nunique()} subjects')
print()

print(f'{"Category":<16} {"Type":<12} {"OTC M":>8} {"OTC SD":>8} {"Ctrl M":>8} {"Ctrl SD":>8}')
print('─' * 64)
for cat in CATEGORIES:
    ov = otc_avg[otc_avg['measured_category'] == cat]['mds_shift'].values
    cv = ctrl_avg[ctrl_avg['measured_category'] == cat]['mds_shift'].values
    lat = 'bilateral' if cat in BILATERAL else 'unilateral'
    print(f'{cat:<16} {lat:<12} {np.nanmean(ov):>8.3f} {np.nanstd(ov):>8.3f} '
          f'{np.nanmean(cv):>8.3f} {np.nanstd(cv):>8.3f}')

# Sym vs asym per patient
print(f'\nPer patient:')
for sub in sorted(otc_avg['subject'].unique()):
    sub_df = otc_avg[otc_avg['subject'] == sub]
    sv = [sub_df[sub_df['measured_category'] == c]['mds_shift'].values for c in BILATERAL]
    av = [sub_df[sub_df['measured_category'] == c]['mds_shift'].values for c in UNILATERAL]
    s = np.nanmean([v[0] for v in sv if len(v)])
    a = np.nanmean([v[0] for v in av if len(v)])
    direction = '↑ sym shifts more' if s > a else '↓ asym shifts more'
    print(f'  {sub}: sym={s:.3f}, asym={a:.3f}, diff={s-a:+.3f} {direction}')

# Group level
sym_vals, asym_vals = [], []
for sub in otc_avg['subject'].unique():
    sub_df = otc_avg[otc_avg['subject'] == sub]
    sv = [sub_df[sub_df['measured_category'] == c]['mds_shift'].values for c in BILATERAL]
    av = [sub_df[sub_df['measured_category'] == c]['mds_shift'].values for c in UNILATERAL]
    s = np.nanmean([v[0] for v in sv if len(v)])
    a = np.nanmean([v[0] for v in av if len(v)])
    if np.isfinite(s) and np.isfinite(a):
        sym_vals.append(s)
        asym_vals.append(a)

sym_arr = np.array(sym_vals)
asym_arr = np.array(asym_vals)
print(f'\nOTC (n={len(sym_arr)}):')
print(f'  Symmetric M  = {sym_arr.mean():.3f} (SD={sym_arr.std():.3f})')
print(f'  Asymmetric M = {asym_arr.mean():.3f} (SD={asym_arr.std():.3f})')
diff = sym_arr.mean() - asym_arr.mean()
print(f'  Diff = {diff:+.3f}')
if len(sym_arr) >= 3:
    t, p = ttest_rel(sym_arr, asym_arr)
    print(f'  Paired t({len(sym_arr)-1})={t:.3f}, p={p:.4f}')

MDS SHIFT (excluding sub-008 and sub-017)
  OTC: 4 subjects
  Controls: 9 subjects

Category         Type            OTC M   OTC SD   Ctrl M  Ctrl SD
────────────────────────────────────────────────────────────────
face             unilateral      0.161    0.023    0.234    0.070
house            bilateral       0.160    0.078    0.202    0.080
object           bilateral       0.108    0.033    0.167    0.065
word             unilateral      0.172    0.099    0.203    0.112

Per patient:
  OTC004: sym=0.210, asym=0.261, diff=-0.050 ↓ asym shifts more
  OTC010: sym=0.147, asym=0.107, diff=+0.040 ↑ sym shifts more
  OTC021: sym=0.110, asym=0.169, diff=-0.059 ↓ asym shifts more
  OTC079: sym=0.069, asym=0.130, diff=-0.060 ↓ asym shifts more

OTC (n=4):
  Symmetric M  = 0.134 (SD=0.052)
  Asymmetric M = 0.167 (SD=0.059)
  Diff = -0.032


NameError: name 'ttest_rel' is not defined

In [ ]:
sel = pd.read_csv(SEL_DIR / 'selectivity_summary.csv')
sel = sel[~sel['sub'].str.contains('017')]
sel['ses_int'] = sel['ses'].astype(int)

otc_sel = sel[sel['group'] == 'OTC'].copy()
otc_sel['surgery_side'] = otc_sel['intact_hemi'].map(
    lambda h: 'right' if h == 'left' else 'left')

# Keep only intact hemisphere
otc_sel = otc_sel[
    ((otc_sel['intact_hemi'] == 'left') & (otc_sel['hemi'] == 'left')) |
    ((otc_sel['intact_hemi'] == 'right') & (otc_sel['hemi'] == 'right'))
]

# Only subjects with 2+ sessions
multi = otc_sel.groupby('sub')['ses_int'].nunique()
multi_subs = multi[multi >= 2].index.tolist()
otc_multi = otc_sel[otc_sel['sub'].isin(multi_subs)]

metric = 'sum_selec_norm'

print('SELECTIVITY CHANGE (last - first session)')
print(f'  Subjects: {sorted(multi_subs)}')
print()

print(f'{"Subject":<12} {"Side":<8} {"Category":<10} {"Ses1":>8} {"SesN":>8} {"Change":>8}')
print('─' * 58)

rows = []
for sub in sorted(multi_subs):
    sub_df = otc_multi[otc_multi['sub'] == sub]
    side = sub_df['surgery_side'].iloc[0]
    ses_list = sorted(sub_df['ses_int'].unique())
    first, last = ses_list[0], ses_list[-1]
    
    for cat in CATEGORIES:
        v1 = sub_df[(sub_df['ses_int'] == first) & (sub_df['category'] == cat)][metric].values
        vn = sub_df[(sub_df['ses_int'] == last) & (sub_df['category'] == cat)][metric].values
        if len(v1) and len(vn):
            change = vn[0] - v1[0]
            lat = 'symmetric' if cat in SYMMETRIC else 'asymmetric'
            print(f'{sub:<12} {side:<8} {cat:<10} {v1[0]:>8.1f} {vn[0]:>8.1f} {change:>8.1f}')
            rows.append({'subject': sub, 'side': side, 'category': cat,
                         'type': lat, 'change': change})

# Symmetric vs asymmetric
print()
rdf = pd.DataFrame(rows)
sym_vals, asym_vals = [], []
for sub in rdf['subject'].unique():
    sd = rdf[rdf['subject'] == sub]
    s = sd[sd['type'] == 'symmetric']['change'].mean()
    a = sd[sd['type'] == 'asymmetric']['change'].mean()
    if np.isfinite(s) and np.isfinite(a):
        sym_vals.append(s)
        asym_vals.append(a)

sym_arr = np.array(sym_vals)
asym_arr = np.array(asym_vals)
m_diff, ci_diff = boot_ci_diff(sym_arr, asym_arr)
print(f'Symmetric vs Asymmetric change (n={len(sym_arr)}):')
print(f'  Symmetric M  = {sym_arr.mean():.1f}  95% CI {[f"{x:.1f}" for x in boot_ci(sym_arr)]}')
print(f'  Asymmetric M = {asym_arr.mean():.1f}  95% CI {[f"{x:.1f}" for x in boot_ci(asym_arr)]}')
print(f'  Diff (sym-asym) = {m_diff:.1f}  95% CI [{ci_diff[0]:.1f}, {ci_diff[1]:.1f}]')
sig = 'significant' if ci_diff[0] > 0 or ci_diff[1] < 0 else 'not significant'
print(f'  → {sig} (positive = symmetric changes more)')

SELECTIVITY CHANGE (last - first session)
  Subjects: ['sub-004', 'sub-008', 'sub-010', 'sub-021', 'sub-079']

Subject      Side     Category       Ses1     SesN   Change
──────────────────────────────────────────────────────────
sub-004      right    face           27.8    443.9    416.1
sub-004      right    house         317.7    728.8    411.1
sub-004      right    object        360.5   1109.1    748.6
sub-004      right    word            3.2     97.5     94.3
sub-008      right    face          966.2    134.8   -831.4
sub-008      right    house         643.2     86.8   -556.4
sub-008      right    object        715.0     19.0   -696.0
sub-008      right    word           18.0     16.1     -1.9
sub-010      left     face           19.5     17.5     -2.1
sub-010      left     house         322.7    112.7   -210.0
sub-010      left     object       1033.1    427.8   -605.3
sub-010      left     word           25.7     18.7     -7.0
sub-021      left     face          388.0    772.1

In [ ]:
print('SELECTIVITY CHANGE by resection side:')
print()
for side in ['left', 'right']:
    side_sym, side_asym = [], []
    side_subs = rdf[rdf['side'] == side]['subject'].unique()
    for sub in side_subs:
        sd = rdf[rdf['subject'] == sub]
        s = sd[sd['type'] == 'symmetric']['change'].mean()
        a = sd[sd['type'] == 'asymmetric']['change'].mean()
        if np.isfinite(s) and np.isfinite(a):
            side_sym.append(s)
            side_asym.append(a)
            print(f'  {sub} ({side}): sym={s:.1f}, asym={a:.1f}, diff={s-a:.1f}')
    
    if len(side_sym) >= 2:
        sa = np.array(side_sym)
        aa = np.array(side_asym)
        m, ci = boot_ci_diff(sa, aa)
        n_pos = sum((sa - aa) > 0)
        print(f'  {side} resection (n={len(sa)}):')
        print(f'    Diff (sym-asym) = {m:.1f}  95% CI [{ci[0]:.1f}, {ci[1]:.1f}]')
        print(f'    Direction: {n_pos}/{len(sa)} sym changes more')
    else:
        print(f'  {side} resection: n={len(side_sym)}, too few')
    print()

SELECTIVITY CHANGE by resection side:

  sub-010 (left): sym=-407.6, asym=-4.5, diff=-403.1
  sub-021 (left): sym=287.2, asym=207.5, diff=79.7
  sub-079 (left): sym=303.3, asym=-131.5, diff=434.8
  left resection (n=3):
    Diff (sym-asym) = 37.1  95% CI [-403.1, 434.8]
    Direction: 2/3 sym changes more

  sub-004 (right): sym=579.8, asym=255.2, diff=324.6
  sub-008 (right): sym=-626.2, asym=-416.6, diff=-209.5
  right resection (n=2):
    Diff (sym-asym) = 57.5  95% CI [-209.5, 324.6]
    Direction: 1/2 sym changes more



In [ ]:
print('LIU DISTINCTIVENESS CHANGE (last - first session)')
print()

rows = []
print(f'{"Subject":<12} {"Side":<8} {"Category":<10} {"Ses1":>8} {"SesN":>8} {"Change":>8}')
print('─' * 58)

for sub in sorted(otc_liu['subject'].unique()):
    sub_df = otc_liu[otc_liu['subject'] == sub]
    sessions = sorted(sub_df['session'].unique())
    if len(sessions) < 2:
        continue
    first, last = sessions[0], sessions[-1]
    side = sub_df['surgery_side'].iloc[0]
    
    for cat in CATEGORIES:
        v1 = sub_df[(sub_df['session'] == first) & (sub_df['category'] == cat)]['liu_distinctiveness'].values
        vn = sub_df[(sub_df['session'] == last) & (sub_df['category'] == cat)]['liu_distinctiveness'].values
        if len(v1) and len(vn):
            change = vn[0] - v1[0]
            lat = 'symmetric' if cat in SYMMETRIC else 'asymmetric'
            print(f'{sub:<12} {side:<8} {cat:<10} {v1[0]:>8.3f} {vn[0]:>8.3f} {change:>8.3f}')
            rows.append({'subject': sub, 'side': side, 'category': cat,
                         'type': lat, 'change': change})

rdf = pd.DataFrame(rows)
print()

# Pooled
sym_vals, asym_vals = [], []
for sub in rdf['subject'].unique():
    sd = rdf[rdf['subject'] == sub]
    s = sd[sd['type'] == 'symmetric']['change'].mean()
    a = sd[sd['type'] == 'asymmetric']['change'].mean()
    if np.isfinite(s) and np.isfinite(a):
        sym_vals.append(s)
        asym_vals.append(a)

sym_arr = np.array(sym_vals)
asym_arr = np.array(asym_vals)
m_diff, ci_diff = boot_ci_diff(sym_arr, asym_arr)
n_pos = sum((sym_arr - asym_arr) > 0)
print(f'Pooled (n={len(sym_arr)}):')
print(f'  Symmetric M  = {sym_arr.mean():.3f}')
print(f'  Asymmetric M = {asym_arr.mean():.3f}')
print(f'  Diff = {m_diff:.3f}  95% CI [{ci_diff[0]:.3f}, {ci_diff[1]:.3f}]')
print(f'  Direction: {n_pos}/{len(sym_arr)} sym worsens more')
print(f'  (positive = symmetric becomes less distinct over time)')
print()

# By resection side
for side in ['left', 'right']:
    s_s, a_s = [], []
    side_subs = rdf[rdf['side'] == side]['subject'].unique()
    for sub in side_subs:
        sd = rdf[rdf['subject'] == sub]
        s = sd[sd['type'] == 'symmetric']['change'].mean()
        a = sd[sd['type'] == 'asymmetric']['change'].mean()
        if np.isfinite(s) and np.isfinite(a):
            s_s.append(s)
            a_s.append(a)
            print(f'  {sub} ({side}): sym={s:.3f}, asym={a:.3f}, diff={s-a:.3f}')
    if len(s_s) >= 2:
        sa, aa = np.array(s_s), np.array(a_s)
        m, ci = boot_ci_diff(sa, aa)
        n_p = sum((sa - aa) > 0)
        print(f'  {side} (n={len(sa)}): diff={m:.3f} CI [{ci[0]:.3f}, {ci[1]:.3f}], {n_p}/{len(sa)} sym>asym')
    print()

LIU DISTINCTIVENESS CHANGE (last - first session)

Subject      Side     Category       Ses1     SesN   Change
──────────────────────────────────────────────────────────
OTC004       right    face          0.424    0.024   -0.400
OTC004       right    house         0.781   -0.277   -1.059
OTC004       right    object        0.734    1.190    0.455
OTC004       right    word          0.439    0.646    0.207
OTC008       right    face          1.108    0.331   -0.776
OTC008       right    house         1.128   -0.080   -1.208
OTC008       right    object        1.321    0.569   -0.752
OTC008       right    word          1.349    0.024   -1.325
OTC010       left     face          0.612    1.046    0.434
OTC010       left     house         0.783    0.440   -0.343
OTC010       left     object        1.449    2.042    0.593
OTC010       left     word          0.304    0.423    0.119
OTC021       left     face          1.224    1.109   -0.115
OTC021       left     house         0.655    0.633

In [ ]:
# Is the sym-asym gap LARGER in OTC than controls?
# OTC diffs (already have these, n=5 excl 017)
otc_diffs = np.array([b - u for b, u, s in zip(bilat_vals, uni_vals, subs_used) 
                       if s != 'OTC017'])

# Control diffs (already computed as ctrl_diffs)
# ctrl_diffs is bil - uni per control hemisphere

print('GAP AMPLIFICATION TEST')
print(f'  OTC diff (sym-asym):     M={otc_diffs.mean():.3f}  n={len(otc_diffs)}')
print(f'  Control diff (sym-asym): M={ctrl_diffs.mean():.3f}  n={len(ctrl_diffs)}')
print()

# Bootstrap the difference-of-differences
n_boot = 100_000
dod = []
for _ in range(n_boot):
    o = rng_boot.choice(otc_diffs, size=len(otc_diffs), replace=True)
    c = rng_boot.choice(ctrl_diffs, size=len(ctrl_diffs), replace=True)
    dod.append(o.mean() - c.mean())
dod = np.array(dod)
ci = np.percentile(dod, [2.5, 97.5])
obs = otc_diffs.mean() - ctrl_diffs.mean()

print(f'  Difference-of-differences (OTC - Control): {obs:.3f}')
print(f'  95% CI: [{ci[0]:.3f}, {ci[1]:.3f}]')
sig = 'significant' if ci[0] > 0 or ci[1] < 0 else 'not significant'
print(f'  → {sig}')
print()

# Also: per-category OTC vs control (is it just house?)
print('PER-CATEGORY: OTC vs Control geometry')
for cat in CATEGORIES:
    otc_v = []
    for sub in sorted(otc['subject'].unique()):
        if 'OTC017' in sub or '017' in sub:
            continue
        sd = otc[otc['subject'] == sub]
        cv = sd[sd['category'] == cat]['geometry_preservation'].values
        if len(cv) and np.isfinite(cv[0]):
            otc_v.append(cv[0])
    
    ctrl_v = []
    for sub in ctrl['subject'].unique():
        for hemi in ['left', 'right']:
            sd = ctrl[(ctrl['subject'] == sub) & (ctrl['hemi_label'] == hemi)]
            cv = sd[sd['category'] == cat]['geometry_preservation'].values
            if len(cv) and np.isfinite(cv[0]):
                ctrl_v.append(cv[0])
    
    otc_a = np.array(otc_v)
    ctrl_a = np.array(ctrl_v)
    
    diff_vals = []
    for _ in range(n_boot):
        o = rng_boot.choice(otc_a, size=len(otc_a), replace=True)
        c = rng_boot.choice(ctrl_a, size=len(ctrl_a), replace=True)
        diff_vals.append(o.mean() - c.mean())
    diff_arr = np.array(diff_vals)
    ci_cat = np.percentile(diff_arr, [2.5, 97.5])
    obs_cat = otc_a.mean() - ctrl_a.mean()
    sig_cat = '*' if ci_cat[0] > 0 or ci_cat[1] < 0 else ''
    
    lat = 'sym' if cat in SYMMETRIC else 'asym'
    print(f'  {cat:<8} ({lat}): OTC={otc_a.mean():.3f} Ctrl={ctrl_a.mean():.3f} '
          f'diff={obs_cat:.3f} CI [{ci_cat[0]:.3f}, {ci_cat[1]:.3f}] {sig_cat}')

GAP AMPLIFICATION TEST
  OTC diff (sym-asym):     M=-0.432  n=5
  Control diff (sym-asym): M=-0.080  n=18

  Difference-of-differences (OTC - Control): -0.352
  95% CI: [-0.608, -0.104]
  → significant

PER-CATEGORY: OTC vs Control geometry
  face     (asym): OTC=0.725 Ctrl=0.698 diff=0.027 CI [-0.248, 0.252] 
  house    (sym): OTC=0.080 Ctrl=0.443 diff=-0.362 CI [-0.797, 0.104] 
  object   (sym): OTC=0.396 Ctrl=0.584 diff=-0.188 CI [-0.495, 0.077] 
  word     (asym): OTC=0.542 Ctrl=0.439 diff=0.103 CI [-0.316, 0.484] 


In [ ]:
# Selectivity — nonOTC
sel = pd.read_csv(SEL_DIR / 'selectivity_summary.csv')
sel['ses_int'] = sel['ses'].astype(int)
first = sel.groupby('sub')['ses_int'].min().reset_index().rename(columns={'ses_int': 'fs'})
sel = sel.merge(first, on='sub')
sel = sel[sel['ses_int'] == sel['fs']]

nonotc_sel = sel[sel['group'] == 'nonOTC'].copy()
nonotc_sel['surgery_side'] = nonotc_sel['intact_hemi'].map(
    lambda h: 'right' if h == 'left' else 'left')
nonotc_intact = nonotc_sel[
    ((nonotc_sel['intact_hemi'] == 'left') & (nonotc_sel['hemi'] == 'left')) |
    ((nonotc_sel['intact_hemi'] == 'right') & (nonotc_sel['hemi'] == 'right'))
]

print(f'nonOTC selectivity: {nonotc_intact["sub"].nunique()} subjects')
for cat in ['face', 'house', 'object', 'word']:
    v = nonotc_intact[nonotc_intact['category'] == cat]['sum_selec_norm'].values
    print(f'  {cat}: M={np.nanmean(v):.1f} SD={np.nanstd(v):.1f} n={len(v)}')

# Liu distinctiveness — nonOTC
liu = pd.read_csv(LIU_DIR / f'liu_distinctiveness_{COPE_SET}.csv')
first = liu.groupby('subject_id')['session'].min().reset_index().rename(columns={'session': 'fs'})
liu = liu.merge(first, on='subject_id')
liu = liu[liu['session'] == liu['fs']]

nonotc_liu = liu[(liu['group'] == 'nonOTC') & (liu['hemi_label'] == 'intact')].copy()
print(f'\nnonOTC distinctiveness: {nonotc_liu["subject"].nunique()} subjects')
for cat in ['face', 'house', 'object', 'word']:
    v = nonotc_liu[nonotc_liu['category'] == cat]['liu_distinctiveness'].values
    print(f'  {cat}: M={np.nanmean(v):.3f} SD={np.nanstd(v):.3f} n={len(v)}')

# Spatial — nonOTC
pk = pd.read_csv(BASE_DIR / 'group_results' / 'peak_coords' / 'peak_coords.csv')
pk['ses_int'] = pk['ses'].astype(int)
first = pk.groupby('sub')['ses_int'].min().reset_index().rename(columns={'ses_int': 'fs'})
pk = pk.merge(first, on='sub')
pk = pk[pk['ses_int'] == pk['fs']]

nonotc_pk = pk[pk['group'] == 'nonOTC'].copy()
nonotc_pk = nonotc_pk[
    ((nonotc_pk['intact_hemi'] == 'left') & (nonotc_pk['hemi'] == 'left')) |
    ((nonotc_pk['intact_hemi'] == 'right') & (nonotc_pk['hemi'] == 'right'))
]
print(f'\nnonOTC spatial: {nonotc_pk["sub"].nunique()} subjects')

nonOTC selectivity: 9 subjects
  face: M=675.2 SD=409.6 n=9
  house: M=707.2 SD=457.2 n=9
  object: M=1612.1 SD=749.8 n=9
  word: M=75.0 SD=34.9 n=9

nonOTC distinctiveness: 9 subjects
  face: M=0.659 SD=0.223 n=9
  house: M=0.334 SD=0.558 n=9
  object: M=1.224 SD=0.302 n=9
  word: M=0.791 SD=0.443 n=9

nonOTC spatial: 9 subjects


In [ ]:
resamp = pd.read_csv(SEL_DIR / 'resamples' / 'sum_selec_norm_resamples.csv')

sel = pd.read_csv(SEL_DIR / 'selectivity_summary.csv')
sel['ses_int'] = sel['ses'].astype(int)
first = sel.groupby('sub')['ses_int'].min().reset_index().rename(columns={'ses_int': 'fs'})
sel = sel.merge(first, on='sub')
sel = sel[sel['ses_int'] == sel['fs']]
sel = sel[~sel['sub'].str.contains('017')]

otc_sel = sel[sel['group'] == 'OTC'].copy()
otc_sel['surgery_side'] = otc_sel['intact_hemi'].map(
    lambda h: 'right' if h == 'left' else 'left')
otc_intact = otc_sel[
    ((otc_sel['intact_hemi'] == 'left') & (otc_sel['hemi'] == 'left')) |
    ((otc_sel['intact_hemi'] == 'right') & (otc_sel['hemi'] == 'right'))
]

PREF = {'face': 'right', 'word': 'left', 'house': 'left', 'object': 'left'}

print('BOOTSTRAP PERCENTILES: OTC patients vs control distribution')
print(f'{"Subject":<12} {"Side":<8} {"Category":<10} {"Value":>8} {"Percentile":>12} {"Below 5%":>10}')
print('─' * 65)

summary = {}
for cat in ['face', 'word', 'object', 'house']:
    pref = PREF[cat]
    col = f'{cat}_{pref}'
    boot_dist = resamp[col].values
    
    below_5 = 0
    above_95 = 0
    total = 0
    
    for _, row in otc_intact[otc_intact['category'] == cat].iterrows():
        val = row['sum_selec_norm']
        pct = np.mean(boot_dist <= val) * 100
        b5 = '*' if pct < 5 else ''
        print(f'{row["sub"]:<12} {row["surgery_side"]:<8} {cat:<10} {val:>8.1f} {pct:>11.1f}% {b5:>10}')
        total += 1
        if pct < 5:
            below_5 += 1
        if pct > 95:
            above_95 += 1
    
    summary[cat] = {'total': total, 'below_5': below_5, 'above_95': above_95}

print()
print('SUMMARY:')
for cat in ['face', 'word', 'object', 'house']:
    s = summary[cat]
    lat = 'asymmetric' if cat in ['face', 'word'] else 'symmetric'
    print(f'  {cat} ({lat}): {s["below_5"]}/{s["total"]} below 5th, '
          f'{s["above_95"]}/{s["total"]} above 95th')

BOOTSTRAP PERCENTILES: OTC patients vs control distribution
Subject      Side     Category      Value   Percentile   Below 5%
─────────────────────────────────────────────────────────────────
sub-004      right    face           27.8         0.0%          *
sub-008      right    face          966.2        97.8%           
sub-010      left     face           19.5         0.0%          *
sub-021      left     face          388.0        34.5%           
sub-066      left     face          276.6        14.3%           
sub-069      left     face           22.6         0.0%          *
sub-074      right    face            3.5         0.0%          *
sub-075      right    face          134.0         1.1%          *
sub-076      right    face          761.0        88.8%           
sub-077      right    face          730.8        86.5%           
sub-078      left     face          347.9        26.8%           
sub-079      left     face          933.5        96.9%           
sub-089      rig